# Pipeline Parallelism
要理解 Pipeline Parallelism（PP，流水线并行）中的通信重叠 及其核心调度策略“1F1B”，我们需要从流水线的瓶颈、调度算法的演进以及 Megatron-LM 的具体实现逻辑三个层面来剖析。

## 一、Pipeline Parallelism (PP) 的背景与痛点
在大规模模型训练中，当单卡显存无法容纳整个模型时，除了 Tensor Parallelism（TP），还需要将模型的不同层（Layers）切分到不同的 GPU 上，这就是 Pipeline Parallelism。

- 基本流程：数据像工厂流水线一样，依次经过 GPU0 -> GPU1 -> ... -> GPUn。
- 核心问题——气泡（Bubble）：在传统的简单调度（如 GPipe）中，GPU 必须等待前一个阶段完成才能开始工作。这导致在训练的开始阶段（Warmup）和结束阶段（Cooldown），大部分 GPU 处于空闲状态，形成了巨大的“气泡”，严重降低了硬件利用率。

为了解决这个问题，需要一种更高效的调度策略，而 1F1B 就是其中的佼佼者。

## 二、什么是 1F1B (One Forward One Backward)？
1F1B 是一种流水线调度策略，全称为“One Forward pass followed by One Backward pass”（一次前向传播紧跟一次反向传播）。

它的核心思想是：一旦某个 GPU 完成了它负责的那个微批次的前向计算，它不等待所有 GPU 都完成前向，而是立即利用刚计算出的激活值进行反向传播。

###### 为什么 1F1B 能天然支持通信重叠？
在 1F1B 模式下，GPU 的执行流变得非常紧凑且规律：
- 计算密集度均匀：相比于 GPipe 的“突发式”计算（先狂算前向，再狂算反向），1F1B 让每个 GPU 在每个时间步都有事做（要么在做前向，要么在做反向）。
- 腾出带宽窗口：由于计算任务被细化并穿插执行，GPU 的计算单元（Tensor Cores）和通信单元（NVLink/InfiniBand）之间的资源竞争变得更加可控。这就为插入通信操作提供了天然的“空隙”。

## 三、Megatron 中 1F1B 的通信重叠机制
图片中提到的“Megatron 的 1F1B 流水线调度天然支持通信与计算重叠”，具体是指以下两个层面的重叠：

### 1. 跨 GPU 的流水线气泡掩盖
这是最宏观的重叠。
- 场景：当 GPU k 正在计算第 i 个微批次的前向传播时，GPU k+1 可能正在计算第 i−1 个微批次的反向传播。
- 重叠效果：虽然物理上是串行的依赖关系，但在时间轴上，不同 GPU 的处理是并行的。更重要的是，当 GPU k 完成前向并将激活值发送给 GPU k+1 后，它可以立即开始处理下一个微批次或反向传播，而不需要等待整个批次结束。这种细粒度的交错大大减少了整体的空闲时间。

### 2. 计算与通信指令级的重叠
这是微观层面的重叠，也是 Megatron 优化的重点。在 1F1B 的每一个步骤中，Megatron 利用了 CUDA 流的特性来实现重叠：
- 发送与计算重叠：
  - 当 GPU 完成了一个微批次的前向计算（Forward Pass），生成了输出激活值。
  - 传统做法：等待数据完全发送到下一张卡，然后再开始下一步。
  - Megatron 做法：启动一个异步的 P2P Send 操作（将激活值发给下游 GPU），然后立即在当前 GPU 上启动下一个微批次的前向计算（或者如果是反向阶段，则启动反向计算）。此时，数据在网卡/NVLink 上传输，而 GPU 核心在计算，两者互不干扰。
- 接收与计算重叠：
  - 同理，当 GPU 需要等待上游传来的输入数据时，它可以先安排一些不依赖该数据的准备工作（如梯度清零、优化器状态更新等），或者如果是在反向传播阶段，利用之前缓存的激活值先进行部分梯度的计算，同时异步等待新的梯度传入。

## 四、总结：1F1B 的优势
相比于其他调度策略（如 GPipe 或 Interleaved 1F1B），标准的 1F1B 在 Megatron 中之所以被强调支持通信重叠，是因为：
- 内存友好：它不需要像 GPipe 那样保存所有微批次的激活值，只需保存当前正在处理的那一个，节省了显存，从而允许使用更大的 Batch Size，间接提高了吞吐率。
- 负载均衡：它消除了大部分的气泡时间，使得通信操作（Send/Recv）可以更均匀地分布在计算间隙中，避免了通信拥堵。
- 实现简洁高效：在代码实现上，1F1B 的逻辑是一个清晰的循环，非常容易在其中插入异步通信原语，使得编译器或运行时更容易优化指令流水线。

简而言之，1F1B 通过“做完一个算一个”的策略，把原本大块的等待时间切碎，填补进了计算任务中，从而让通信操作能够见缝插针地与计算并行执行。

# 实现

在 Megatron-LM 中，Pipeline Parallelism (PP) 的 1F1B 调度及其通信重叠机制主要在以下核心文件中实现：
- megatron/core/pipeline_parallel/schedules.py：
  - 这是 PP 调度的核心文件，包含了 forward_backward_pipelining_with_1f1b 函数。该函数实现了标准的 1F1B 调度逻辑，包括 warmup、steady state（稳定状态）和 cooldown 三个阶段。在该文件中，代码通过控制前向传播和反向传播的执行顺序，使得 GPU 在完成一个微批次的前向后立即开始反向传播，从而为通信操作腾出时间窗口，实现计算与通信的重叠。
- megatron/core/pipeline_parallel/p2p_communication.py：
  - 该文件封装了流水线并行中所需的点对点通信原语，如 send_forward、recv_forward、send_backward、recv_backward。这些函数负责在不同 GPU 之间传递激活值和梯度。在 1F1B 调度中，这些通信操作被巧妙地插入到前向和反向计算的间隙中，利用 CUDA 流的异步特性实现重叠。
- megatron/core/transformer/transformer_block.py：
  - 虽然此文件主要负责模型结构的定义，但它也参与了 PP 的切分逻辑。例如，get_num_layers_to_build 函数决定了每个 GPU 需要构建的 Transformer 层数，这直接影响了流水线的划分和后续调度的执行。

## 实现逻辑简述
在上述文件中，1F1B 调度的实现通常遵循以下步骤：
- Warmup 阶段：前几个微批次只执行前向传播，以填充流水线，确保后续阶段有数据可处理。
- Steady State 阶段：进入 1F1B 模式，每个 GPU 交替执行一个前向和一个反向。此时，前向计算完成后立即启动反向计算，而通信操作（如发送激活值或接收梯度）则被安排在计算的空隙中，利用非阻塞通信和 CUDA 流实现重叠。
- Cooldown 阶段：最后几个微批次只执行反向传播，以完成所有梯度的计算和回传。
这种设计使得 Megatron-LM 能够在保持高计算利用率的同时，有效隐藏通信延迟，提升整体训练效率。